# 13 — HMMs and Influence Models as generative models

A generative model is a probability recipe capable of making complete synthetic data. An HMM does not merely classify a sequence: it tells a story for how one could have been produced.

$$z_1\sim\pi,\qquad x_t\sim B_{z_t},\qquad z_{t+1}\sim A_{z_t}.$$

Multiplying those local choices gives a joint probability $p(x_{1:T},z_{1:T})$. Summing over hidden stories gives $p(x_{1:T})$; conditioning the joint gives $p(z_{1:T}\mid x_{1:T})$. Generation, scoring, and inference are three views of one joint distribution.

In [ ]:
import torch

from latentlinks.hmm import CategoricalHMM, forward_log_likelihood

hmm = CategoricalHMM(
    torch.tensor([0.8, 0.2]),
    torch.tensor([[0.95, 0.05], [0.15, 0.85]]),
    torch.tensor([[0.9, 0.1], [0.1, 0.9]]),
)
for seed in range(3):
    observations, states = hmm.sample(12, seed=seed)
    score = round(float(forward_log_likelihood(hmm, observations)), 3)
    print("z:", states.tolist(), "x:", observations.tolist(), "log p(x):", score)

## What generation is useful for

- **Prior prediction:** continue a latent process before seeing new data.
- **Posterior predictive checks:** fit a model, simulate from it, and compare durations, symbol frequencies, and cross-chain lags with real data.
- **Counterfactual model experiments:** change an influence weight and inspect the model-implied result. This probes the fitted model, not necessarily a real-world causal intervention.
- **Recovery tests:** plant known parameters, generate data, fit blindly, and measure what can actually be recovered.

In an Influence Model, each target mixes proposals from source chains: $p(z_{t+1,c}=s\mid z_t)=\sum_jD_{c,j}A_{c,j,z_{t,j},s}$. Sampling from this mixture is the generative semantics—not an afterthought.

In [ ]:
from latentlinks.influence import mit_six_chain_model

influence_model = mit_six_chain_model()
x, z = influence_model.sample(20, seed=13)
print("observation shape:", tuple(x.shape), "latent shape:", tuple(z.shape))
print("first latent configuration:", z[0].tolist())
assert x.shape == z.shape == (20, 6)